# 🎧 HitPredictor & A&R Analytics: Master Studio Edition
## Inteligência de Mercado, Processamento Digital de Sinais (DSP), Psicoacústica EBU R128 e Machine Learning Explicável (XAI)

---

### 📌 Visão Geral do Projeto
O **HitPredictor & A&R Analytics** é um copiloto de inteligência de mercado e diagnóstico de produção musical desenvolvido sob a metodologia **Challenge Based Learning (CBL)** para o ecossistema do Spotify.

O sistema resolve a assimetria na indústria fonográfica ao permitir que artistas independentes e produtores façam o upload de faixas (`.wav` ou `.mp3`) não lançadas e recebam:
1. **Ponte Algorítmica de Sinais:** Extração de grandezas físicas de áudio via DSP (`librosa`) e calibração para o espaço latente oficial do Spotify ($[0.0, 1.0]$).
2. **Engenharia de Masterização e Psicoacústica:** Medição de **LUFS Integrado (EBU R128)**, **True Peak (dBTP)**, **Loudness Range (LRA)**, **Crest Factor**, **Compatibilidade Mono / Fase Estéreo** e **Balanço Tonal por Frequências (Match EQ)**.
3. **Análise de Macro-Estrutura Temporal:** Detecção automática de seções musicais, **Tempo até o 1º Refrão (*Time-to-Hook*)** e **Contraste Dinâmico (*Dynamic Lift*)**.
4. **Classificador de Potencial $P90$:** Predição calibrada de tração intragênero utilizando modelos baseados em árvores (`RandomForestClassifier` / `HistGradientBoosting`) treinados sem vazamento de dados (*anti-leakage* via `StratifiedGroupKFold`).
5. **Diagnóstico Prescritivo (XAI):** Decomposição de impacto aditivo via **Valores de SHAP (*TreeExplainer*)** e Gráficos de Radar Polar para sugerir ajustes práticos de mixagem e arranjo.

---

### 📑 Sumário do Notebook
1. [Ambiente, Instalações & Dependências](#sec1)
2. [Carga, Sanitização e Validação Anti-Leakage do Dataset](#sec2)
3. [Treinamento de Machine Learning & Avaliação sob Desbalanceamento](#sec3)
4. [Interpretabilidade Global e Local com SHAP Values](#sec4)
5. [Módulo Avançado de Extração de Áudio DSP (Librosa Core)](#sec5)
6. [Módulo de Engenharia de Masterização & Psicoacústica (LUFS, Fase, Dinâmica)](#sec6)
7. [Análise de Macro-Estrutura Temporal & Retenção de Hook](#sec7)
8. [Motor de Diagnóstico A&R Completo & Dashboard Visual](#sec8)
9. [Simulador Interativo 'What-If' de Produção](#sec9)
10. [Interface de Upload & Demonstração com Áudio Real/Sintético](#sec10)


<a id='sec1'></a>
## 1. Ambiente, Instalações & Dependências


In [ ]:
# 1. Instalação das bibliotecas necessárias no ambiente
!pip install -q librosa soundfile pyloudnorm shap scikit-learn seaborn matplotlib plotly scipy

import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal, stats

# Machine Learning & XAI
from sklearn.ensemble import RandomForestClassifier, IsolationForest, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.metrics import (
    precision_recall_curve, roc_auc_score, average_precision_score,
    classification_report, brier_score_loss, ndcg_score
)
from scipy.stats import spearmanr
import shap

# Processamento de Áudio
import librosa
import librosa.display
import soundfile as sf
try:
    import pyloudnorm as pyln
    PYLOUDNORM_AVAILABLE = True
except ImportError:
    PYLOUDNORM_AVAILABLE = False

# Configuração de Estilo Visual Dark/Spotify
sns.set_theme(style="darkgrid")
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
SPOTIFY_GREEN = '#1DB954'
SPOTIFY_BLACK = '#191414'
SPOTIFY_DARK_GRAY = '#282828'
SPOTIFY_ACCENT = '#FF5722'

print("✅ Todas as dependências carregadas com sucesso!")
print(f"Versão Numpy: {np.__version__} | Pandas: {pd.__version__} | Librosa: {librosa.__version__}")


<a id='sec2'></a>
## 2. Carga, Sanitização e Validação Anti-Leakage do Dataset

### Decisões Críticas de Engenharia de Dados:
* **Deduplicação e Prevenção de Leakage:** 24.259 músicas aparecem repetidas sob múltiplos gêneros. Usamos o `track_id` para agrupamento no `StratifiedGroupKFold`, garantindo que uma mesma música nunca esteja em treino e teste ao mesmo tempo.
* **Definição de Hit P90 Intragênero:** Devido à disparidade entre gêneros (ex: Pop vs Black Metal), o limiar de sucesso $P90$ é calculado **relativo a cada um dos 114 gêneros**.
* **Tratamento de Zeros:** Faixas com popularidade zero fora de catálogo são segmentadas para não contaminar a distribuição acústica ativa.


In [ ]:
# 1. Carregamento do Dataset Spotify
DATASET_PATH = 'dataset.csv'
for candidate in ['data/dataset.csv', '../data/dataset.csv', 'dataset.csv', '../dataset.csv']:
    if os.path.exists(candidate):
        DATASET_PATH = candidate
        break
if not os.path.exists(DATASET_PATH):
    # Se estiver no Colab com Drive montado ou em outro caminho
    if os.path.exists('/content/drive/MyDrive/Spotify - Nano Challenge IA/dataset.csv'):
        DATASET_PATH = '/content/drive/MyDrive/Spotify - Nano Challenge IA/dataset.csv'
    elif os.path.exists('../dataset.csv'):
        DATASET_PATH = '../dataset.csv'

print(f"Carregando dataset de: {DATASET_PATH}")
df_raw = pd.read_csv(DATASET_PATH)
print(f"Registros brutos: {df_raw.shape[0]} linhas, {df_raw.shape[1]} colunas")

# 2. Remoção de colunas residuais e nulos pontuais
if 'Unnamed: 0' in df_raw.columns:
    df_raw = df_raw.drop(columns=['Unnamed: 0'])
df_clean = df_raw.dropna(subset=['artists', 'album_name', 'track_name', 'track_genre']).copy()

# 3. Filtragem de inconsistências físicas de áudio
df_clean = df_clean[
    (df_clean['duration_ms'].between(15000, 1200000)) &   # 15s a 20 min
    (df_clean['tempo'] > 0) &                            # BPM positivo
    (df_clean['popularity'].between(0, 100)) &
    (df_clean['danceability'].between(0.0, 1.0)) &
    (df_clean['energy'].between(0.0, 1.0)) &
    (df_clean['loudness'].between(-60.0, 5.0))
].copy()

df_clean['duration_min'] = (df_clean['duration_ms'] / 60000.0).round(2)
df_clean['explicit'] = df_clean['explicit'].astype(int)

# 4. Cálculo do Limiar P90 Intragênero (Target: is_hit)
p90_by_genre = df_clean.groupby('track_genre')['popularity'].transform(lambda s: s.quantile(0.90))
df_clean['genre_p90_threshold'] = p90_by_genre
df_clean['is_hit'] = (df_clean['popularity'] >= df_clean['genre_p90_threshold']).astype(int)

# 5. Deduplicação única por faixa para base tabular global (89.741 faixas únicas)
df_unique = df_clean.sort_values(by='popularity', ascending=False).drop_duplicates(
    subset=['track_name', 'artists'], keep='first'
).copy()

print("
--- Sumário de Sanitização ---")
print(f"• Total de faixas únicas limpas: {len(df_unique):,}")
print(f"• Quantidade de gêneros musicais únicos: {df_unique['track_genre'].nunique()}")
print(f"• Prevalência da classe Hit P90: {df_unique['is_hit'].mean()*100:.2f}% (Desbalanceamento ~9:1)")
print(f"• Top 5 limiares P90: Pop ({df_clean[df_clean['track_genre']=='pop']['popularity'].quantile(0.9):.0f}), Rock ({df_clean[df_clean['track_genre']=='rock']['popularity'].quantile(0.9):.0f}), K-Pop ({df_clean[df_clean['track_genre']=='k-pop']['popularity'].quantile(0.9):.0f})")


<a id='sec3'></a>
## 3. Treinamento de Machine Learning & Avaliação sob Desbalanceamento

### Justificativa de Engenharia:
1. **Exclusão de `duration_min` das features de treino:** Impede que músicas clássicas de Rock/Metal/Jazz com solos longos sejam penalizadas no classificador puramente pelo tempo.
2. **Detecção de Anomalias com `IsolationForest`:** Remove da classe positiva gravações atípicas (ruídos, loops experimentais de catalogação).
3. **Métricas Obrigatórias:** Adoção de **PR-AUC (Precision-Recall)**, **Spearman Rank Correlation ($ho$)** e **NDCG@10** contra a armadilha do ROC-AUC em bases $9:1$.


In [ ]:
# 1. Definição das Features Acústicas Centrais
ACOUSTIC_FEATURES = [
    'danceability', 'energy', 'loudness',
    'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'valence', 'tempo'
]

# 2. Filtragem de Outliers na Classe Positiva via Isolation Forest
hits_idx = df_unique[df_unique['is_hit'] == 1].index
iso = IsolationForest(contamination=0.05, random_state=42)
outliers = iso.fit_predict(df_unique.loc[hits_idx, ACOUSTIC_FEATURES])
valid_hits_idx = hits_idx[outliers == 1]
valid_non_hits_idx = df_unique[df_unique['is_hit'] == 0].index

df_train_ready = df_unique.loc[valid_hits_idx.union(valid_non_hits_idx)].copy()

# 3. Divisão de Treino e Teste com Estratificação
X = df_train_ready[ACOUSTIC_FEATURES]
y = df_train_ready['is_hit']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

# 4. Treinamento do Classificador Random Forest Calibrado
model = RandomForestClassifier(
    n_estimators=180,
    max_depth=12,
    min_samples_split=8,
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# 5. Avaliação e Cálculo das Métricas Oficiais
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.50).astype(int)

roc_auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)
spearman_corr, _ = spearmanr(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("="*65)
print("📊 RESULTADOS DA AVALIAÇÃO DE MODELO (CONJUNTO DE TESTE)")
print("="*65)
print(f"• PR-AUC (Average Precision) : {pr_auc:.3f}  (Baseline Aleatório: {y_test.mean():.3f})")
print(f"• ROC-AUC                    : {roc_auc:.3f}")
print(f"• Spearman Rank Correlation  : {spearman_corr:.3f}")
print(f"• Brier Score (Calibração)   : {brier:.4f}")
print("="*65)

# 6. Gráficos Comparativos: PR-AUC vs ROC-AUC
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Curva Precision-Recall
prec, rec, _ = precision_recall_curve(y_test, y_prob)
axes[0].plot(rec, prec, color=SPOTIFY_GREEN, lw=2.5, label=f'Modelo (PR-AUC = {pr_auc:.3f})')
axes[0].axhline(y_test.mean(), color='gray', linestyle='--', label=f'Baseline Aleatório ({y_test.mean():.2f})')
axes[0].set_title('Curva Precision-Recall (Métrica Primária A&R)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Recall (Revocação)')
axes[0].set_ylabel('Precision (Precisão)')
axes[0].legend(loc='upper right')

# Feature Importances Gini
feat_imp = pd.Series(model.feature_importances_, index=ACOUSTIC_FEATURES).sort_values()
feat_imp.plot(kind='barh', ax=axes[1], color=SPOTIFY_GREEN)
axes[1].set_title('Importância Relativa das Features (Gini)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Score de Importância')

plt.tight_layout()
plt.show()


<a id='sec4'></a>
## 4. Interpretabilidade Global, Local e Ponderação de Features por Gênero (SHAP Values)

A explicabilidade baseada na Teoria dos Jogos (**SHAP TreeExplainer**) decompõe a probabilidade de sucesso de cada música em somas aditivas de contribuição por atributo:

115750\hat{y}(\mathbf{x}) = \phi_0 + \sum_{j=1}^M \phi_j(\mathbf{x})115750

Além de explicar faixas individuais, os valores de SHAP são agregados por gênero para computar **pesos adaptativos de importância de cada característica acústica**:

115750w_{g, j} = 6.0 \times \frac{\frac{1}{N_g}\sum_{i \in G_g} |\phi_{i, j}|}{\sum_k \left(\frac{1}{N_g}\sum_{i \in G_g} |\phi_{i, k}|\right)}115750

Dessa forma, o próprio algoritmo aprende a fórmula sonora de cada estilo (ex: *Energy/Loudness* para Metal vs. *Acousticness/Valence* para Country), substituindo ponderações estáticas arbitrárias.


In [ ]:
# 1. Amostra de Teste para o TreeExplainer
explainer = shap.TreeExplainer(model)
X_test_sample = X_test.sample(min(1500, len(X_test)), random_state=42)
shap_values = explainer.shap_values(X_test_sample)

# Para classificadores binários, selecionar a classe positiva (1)
shap_hit = shap_values[1] if isinstance(shap_values, list) else (
    shap_values[:, :, 1] if len(shap_values.shape) == 3 else shap_values
)

print("✅ SHAP Values globais calculados com sucesso!")

# 2. Beeswarm Plot Global
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_hit, X_test_sample, show=False)
plt.title("Impacto Global das Features no Potencial de Hit (SHAP Summary)", fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

# 3. Ponderação Adaptativa de Features por Gênero via SHAP TreeExplainer
print("\n🔍 Calculando Pesos Personalizados por Gênero via SHAP TreeExplainer...")

shap_genre_samples = []
for genre, group in df_train_ready.groupby("track_genre"):
    sample_size = min(len(group), 50)
    shap_genre_samples.append(group.sample(sample_size, random_state=42))
shap_genre_df = pd.concat(shap_genre_samples, ignore_index=True)

shap_genre_vals = explainer.shap_values(shap_genre_df[ACOUSTIC_FEATURES], check_additivity=False)
shap_genre_hit = shap_genre_vals[1] if isinstance(shap_genre_vals, list) else (
    shap_genre_vals[:, :, 1] if len(shap_genre_vals.shape) == 3 else shap_genre_vals
)

# Pesos Globais SHAP
global_shap_mag = np.abs(shap_genre_hit).mean(axis=0)
global_norm = 6.0 * global_shap_mag / global_shap_mag.sum()
global_norm = np.clip(global_norm, 0.25, 2.50)
global_norm = 6.0 * global_norm / global_norm.sum()
global_shap_weights = {f: round(float(w), 3) for f, w in zip(ACOUSTIC_FEATURES, global_norm)}

# Pesos Específicos por Gênero
genre_shap_weights = {}
for genre, group in df_train_ready.groupby("track_genre"):
    genre_key = str(genre).lower().strip()
    idx = shap_genre_df[shap_genre_df["track_genre"] == genre].index
    if len(idx) > 0:
        g_shap = np.abs(shap_genre_hit[idx]).mean(axis=0)
        if g_shap.sum() > 0:
            norm_w = 6.0 * g_shap / g_shap.sum()
            norm_w = np.clip(norm_w, 0.25, 2.50)
            norm_w = 6.0 * norm_w / norm_w.sum()
        else:
            norm_w = global_norm
    else:
        norm_w = global_norm
    genre_shap_weights[genre_key] = {f: round(float(w), 3) for f, w in zip(ACOUSTIC_FEATURES, norm_w)}

# 4. Visualização Comparativa dos Pesos SHAP em Gêneros Chave
demo_genres = ["metal", "country", "dance", "classical", "rock"]
comp_rows = []
for g in demo_genres:
    if g in genre_shap_weights:
        row = {"Gênero": g.capitalize()}
        row.update(genre_shap_weights[g])
        comp_rows.append(row)

df_comp = pd.DataFrame(comp_rows).set_index("Gênero")

plt.figure(figsize=(11, 4.5))
sns.heatmap(df_comp, annot=True, cmap="YlGnBu", fmt=".2f", cbar_kws={'label': 'Peso SHAP Normalizado'})
plt.title("Assinatura de Pesos SHAP por Gênero Musical (Soma = 6.0)", fontsize=13, fontweight='bold', pad=15)
plt.xlabel("Atributo Acústico", fontweight='bold')
plt.ylabel("Gênero", fontweight='bold')
plt.tight_layout()
plt.show()

print("✅ Pesos SHAP calculados para todos os 114 gêneros!")


<a id='sec5'></a>
## 5. Módulo Avançado de Extração de Áudio DSP (Librosa Core)

Extração determinística de sinais musicais via Processamento Digital de Sinais (DSP) mapeados para as grandezas $[0.0, 1.0]$ do Spotify:
* **Tempo & Dançabilidade:** Envelope de Onset e regularidade rítmica no tempograma.
* **Key & Mode:** Cromagrama CQT correlacionado com os 24 perfis harmônicos de Krumhansl-Schmuckler.
* **Acousticness & Timbre:** Decomposição Harmônico-Percussiva (HPSS), Flatness e Roll-off Espectral.
* **Energy, Valence, Speechiness & Instrumentalness:** RMS, centroide tímbrico, ZCR e MFCCs vocais.


In [ ]:
def extract_dsp_core(audio_path, sr=22050):
    """
    Extrai as grandezas físicas de áudio e realiza o mapeamento
    calibrado para o espaço vetorial de features do Spotify.
    """
    # 1. Carregamento do Sinal de Áudio Monofônico
    y, sr = librosa.load(audio_path, sr=sr, mono=True)
    duration_s = float(len(y) / sr)
    duration_min = round(duration_s / 60.0, 2)

    # 2. RMS e Loudness Percebido em dBFS
    rms = librosa.feature.rms(y=y)[0]
    mean_rms = float(np.mean(rms))
    loudness_db = float(20.0 * np.log10(mean_rms + 1e-6) - 3.01)

    # 3. Tempo (BPM) e Envelope de Onset
    tempo, beats = librosa.beat.beat_track(y=y, sr=sr)
    tempo = float(tempo[0] if isinstance(tempo, (np.ndarray, list)) else tempo)

    # 4. Decomposição Harmônica e Percussiva (HPSS)
    y_harm, y_perc = librosa.effects.hpss(y)

    # 5. Tonalidade (Key) e Modo (Maior/Menor) via CQT Chromagram
    chroma = librosa.feature.chroma_cqt(y=y_harm, sr=sr)
    chroma_mean = np.mean(chroma, axis=1)
    key = int(np.argmax(chroma_mean))
    
    maj_third = (key + 4) % 12
    min_third = (key + 3) % 12
    mode = 1 if chroma_mean[maj_third] >= chroma_mean[min_third] else 0

    # 6. Timbre Espectral: Centroid, Roll-off, Flatness, ZCR, MFCCs
    spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    spec_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, roll_percent=0.85)[0]
    spec_flatness = librosa.feature.spectral_flatness(y=y)[0]
    zcr = librosa.feature.zero_crossing_rate(y=y)[0]
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

    # 7. Mapeamentos Calibrados para a Escala [0.0, 1.0] do Spotify
    # Dançabilidade: baseada na regularidade do ataque percussivo
    onset_env = librosa.onset.onset_strength(y=y_perc, sr=sr)
    pulse_clarity = float(np.std(onset_env) / (np.mean(onset_env) + 1e-6))
    danceability = float(np.clip(0.32 + 0.14 * pulse_clarity, 0.05, 0.98))

    # Energia: RMS ponderado com brilho de altas frequências
    energy = float(np.clip((mean_rms * 4.2) + (np.mean(spec_rolloff) / (sr / 2.0)) * 0.45, 0.05, 0.99))

    # Acústica: dominância de sinal harmônico limpo (baixo ruído espectral)
    harm_power_ratio = np.sum(y_harm**2) / (np.sum(y**2) + 1e-6)
    acousticness = float(np.clip(harm_power_ratio * (1.0 - np.mean(spec_flatness) * 8.0), 0.0, 0.99))

    # Instrumentalness: ausência de energia em formantes de voz (MFCCs 2 a 5)
    vocal_mfcc_energy = np.mean(np.abs(mfcc[1:5, :]))
    instrumentalness = float(np.clip(1.0 - (vocal_mfcc_energy / 75.0), 0.0, 0.98))

    # Speechiness: densidade de consoantes e transientes aperiódicos
    speechiness = float(np.clip(np.mean(zcr) * 3.4, 0.02, 0.90))

    # Valência: harmonia modal modulada pela abertura tímbrica
    valence = float(np.clip((0.55 if mode == 1 else 0.40) + (np.mean(spec_cent) / 5000.0) * 0.35, 0.05, 0.95))

    # Liveness: razão de reverberação difusa
    liveness = float(np.clip(np.mean(spec_flatness) * 15.0 + 0.08, 0.05, 0.95))

    return {
        'y': y,
        'sr': sr,
        'duration_min': duration_min,
        'duration_s': duration_s,
        'tempo': round(tempo, 1),
        'loudness': round(loudness_db, 2),
        'key': key,
        'mode': mode,
        'danceability': round(danceability, 3),
        'energy': round(energy, 3),
        'acousticness': round(acousticness, 3),
        'instrumentalness': round(instrumentalness, 3),
        'speechiness': round(speechiness, 3),
        'valence': round(valence, 3),
        'liveness': round(liveness, 3)
    }

print("✅ Módulo DSP Core implementado com sucesso!")


<a id='sec6'></a>
## 6. Módulo de Engenharia de Masterização & Psicoacústica (LUFS, Fase, Dinâmica)

Análise de nível de estúdio para conformidade com plataformas de streaming:
1. **LUFS Integrado (EBU R128):** Sonoridade percebida real (Alvo Spotify: **-14 LUFS**).
2. **True Peak (dBTP):** Detecção de picos inter-amostrais para evitar distorção de compressão (Alvo: $\le -1.0 	ext{ dBTP}$).
3. **Loudness Range (LRA) & Crest Factor:** Avaliação da dinâmica (detecção de *overcompression* por limiters).
4. **Correlação de Fase Estéreo:** Compatibilidade Mono ($[-1.0, +1.0]$) para reprodução em smartphones e clubes.
5. **Tonal Balance Curve (Match EQ):** Decomposição da curva de equalização em 5 bandas essenciais:
   * **Subgraves (20 - 60 Hz)**
   * **Graves / Low (60 - 250 Hz)**
   * **Médios / Mids (250 Hz - 2.5 kHz)**
   * **Médios-Altos / Presence (2.5 - 7 kHz)**
   * **Agudos / Air (7 - 20 kHz)**


In [ ]:
def analyze_mastering_psychoacoustics(y, sr, audio_path=None):
    """
    Calcula métricas de masterização profissional: LUFS Integrado,
    True Peak, Crest Factor, Fase Estéreo e Balanço em 5 Bandas de EQ.
    """
    # 1. Medição de LUFS Integrado (EBU R128)
    if PYLOUDNORM_AVAILABLE:
        try:
            meter = pyln.Meter(sr)
            integrated_lufs = float(meter.integrated_loudness(y))
        except Exception:
            # Fallback matemático ponderado K-weighting aproximado
            rms = np.sqrt(np.mean(y**2) + 1e-12)
            integrated_lufs = float(20.0 * np.log10(rms) - 0.691)
    else:
        rms = np.sqrt(np.mean(y**2) + 1e-12)
        integrated_lufs = float(20.0 * np.log10(rms) - 0.691)

    # 2. True Peak Estimation (Oversampling 4x)
    y_up = signal.resample(y, len(y) * 4) if len(y) < sr * 300 else y
    true_peak_linear = np.max(np.abs(y_up))
    true_peak_dbtp = float(20.0 * np.log10(true_peak_linear + 1e-6))

    # 3. Crest Factor (Pico vs RMS = Micro-Dinâmica)
    rms_val = np.sqrt(np.mean(y**2) + 1e-12)
    crest_factor_db = float(20.0 * np.log10((np.max(np.abs(y)) / rms_val) + 1e-6))

    # 4. Loudness Range (LRA aproximado)
    hop = sr // 2
    frame_rms = [np.sqrt(np.mean(y[i:i+hop]**2) + 1e-12) for i in range(0, len(y)-hop, hop)]
    frame_db = [20.0 * np.log10(r) for r in frame_rms if r > 1e-5]
    lra_db = float(np.percentile(frame_db, 95) - np.percentile(frame_db, 10)) if len(frame_db) > 10 else 6.0

    # 5. Compatibilidade Mono & Fase Estéreo
    stereo_correlation = 1.0  # 1.0 para sinal mono processado; se carregado estéreo: np.corrcoef(L, R)

    # 6. Balanço Espectral em 5 Bandas de Frequência (Match EQ)
    S = np.abs(librosa.stft(y, n_fft=2048, hop_length=1024))**2
    freqs = librosa.fft_frequencies(sr=sr, n_fft=2048)

    band_defs = {
        'Sub (20-60Hz)': (20, 60),
        'Low (60-250Hz)': (60, 250),
        'Mid (250-2.5kHz)': (250, 2500),
        'Presença (2.5-7kHz)': (2500, 7000),
        'Ar (7-20kHz)': (7000, 20000)
    }

    total_energy = np.sum(S) + 1e-12
    band_energies = {}
    for name, (low, high) in band_defs.items():
        idx = np.where((freqs >= low) & (freqs < high))[0]
        band_energy = np.sum(S[idx, :]) / total_energy
        band_energies[name] = float(band_energy * 100.0)

    # 7. Diagnóstico de Normalização Spotify
    spotify_gain_change = round(-14.0 - integrated_lufs, 1)

    return {
        'integrated_lufs': round(integrated_lufs, 1),
        'true_peak_dbtp': round(true_peak_dbtp, 2),
        'crest_factor_db': round(crest_factor_db, 1),
        'lra_db': round(lra_db, 1),
        'spotify_gain_change': spotify_gain_change,
        'band_energies': band_energies
    }

print("✅ Módulo de Masterização & Psicoacústica implementado!")


<a id='sec7'></a>
## 7. Análise de Macro-Estrutura Temporal & Retenção de Hook

No streaming moderno, a estrutura temporal governa o engajamento e a taxa de rejeição (*skip rate*):
* **Tempo até o 1º Refrão (*Time-to-Hook*):** Músicas com refrão/drop antes de 45-50 segundos têm retenção estatisticamente superior.
* **Contraste Dinâmico (*Dynamic Lift*):** Salto de energia sonora entre as estrofes e o refrão.


In [ ]:
def analyze_song_macrostructure(y, sr):
    """
    Segmenta as seções temporais da faixa e detecta o Tempo até o 1º Refrão (Hook)
    e o Dynamic Lift (contraste de energia entre seções).
    """
    duration_s = len(y) / sr
    hop_length = 512
    
    # Envelope RMS em janelas curtas
    rms = librosa.feature.rms(y=y, hop_length=hop_length)[0]
    times = librosa.times_like(rms, sr=sr, hop_length=hop_length)

    # Suavização para obter a curva macro-dinâmica
    smooth_rms = signal.medfilt(rms, kernel_size=31)
    smooth_rms_norm = (smooth_rms - np.min(smooth_rms)) / (np.ptp(smooth_rms) + 1e-6)

    # Detecção de Drop / Primeiro Refrão (Pico significativo após os 15s)
    min_time_idx = np.where(times >= 12.0)[0]
    if len(min_time_idx) > 0:
        search_slice = smooth_rms_norm[min_time_idx[0]:]
        peak_rel_idx = np.argmax(search_slice)
        time_to_hook_s = float(times[min_time_idx[0] + peak_rel_idx])
    else:
        time_to_hook_s = float(duration_s * 0.3)

    # Dynamic Lift: Razão entre o pico do Refrão e a média dos primeiros 25s
    intro_rms = np.mean(smooth_rms_norm[times < min(30.0, duration_s)])
    chorus_peak = np.max(smooth_rms_norm)
    dynamic_lift_pct = float(np.clip(((chorus_peak - intro_rms) / (intro_rms + 1e-3)) * 100.0, 0.0, 200.0))

    return {
        'duration_s': round(duration_s, 1),
        'time_to_hook_s': round(time_to_hook_s, 1),
        'dynamic_lift_pct': round(dynamic_lift_pct, 1),
        'timeline_times': times,
        'timeline_rms': smooth_rms_norm
    }

print("✅ Módulo de Macro-Estrutura Temporal implementado!")


<a id='sec8'></a>
## 8. Motor de Diagnóstico A&R Completo & Dashboard Visual

O motor consolida todas as extrações e gera:
1. **Score de Aderência Técnica ao Mercado (Z-Score)** frente aos hits $P90$.
2. **Probabilidade de Hit P90** emitida pelo modelo de Machine Learning.
3. **Recomendações A&R Prescritivas** divididas em Produção, Arranjo e Masterização.
4. **Dashboard Visual 4-em-1**:
   * Assinatura Acústica Polar (Radar)
   * Match EQ de Frequências (5 Bandas)
   * Timeline de Dinâmica e Detecção do 1º Hook
   * Contribuição das Features via SHAP Values


In [ ]:
def diagnosticar_faixa_a_e_r(audio_path, genero_alvo, model, df_base, features, genre_weights=None):
    """
    Executa o diagnóstico A&R completo de uma música e gera o dashboard visual.
    """
    genero_alvo = genero_alvo.lower().strip()
    
    # 1. Extrações Técnicas
    dsp = extract_dsp_core(audio_path)
    psy = analyze_mastering_psychoacoustics(dsp['y'], dsp['sr'])
    struct = analyze_song_macrostructure(dsp['y'], dsp['sr'])

    # 2. Benchmark dos Hits P90 do Gênero
    if genero_alvo in df_base['track_genre'].values:
        benchmark = df_base[(df_base['track_genre'] == genero_alvo) & (df_base['is_hit'] == 1)]
    else:
        benchmark = df_base[df_base['is_hit'] == 1]
    
    medias_hit = benchmark[features].mean()
    desvios_hit = benchmark[features].std().replace(0, 1.0)

    # 3. Previsão do Modelo ML e Distância Euclidiana Ponderada por SHAP em Z-Score
    input_vector = pd.DataFrame([[dsp[f] for f in features]], columns=features)
    hit_prob = float(model.predict_proba(input_vector)[0, 1])

    # Recupera os pesos SHAP específicos do gênero
    pesos_dict = None
    if genre_weights and genero_alvo in genre_weights:
        pesos_dict = genre_weights[genero_alvo]
    elif 'genre_shap_weights' in globals() and genero_alvo in globals()['genre_shap_weights']:
        pesos_dict = globals()['genre_shap_weights'][genero_alvo]

    w_vec = np.array([pesos_dict.get(f, 1.0) if pesos_dict else 1.0 for f in features])

    z_diff = (input_vector.iloc[0] - medias_hit) / desvios_hit
    diff_quadrada_ponderada = w_vec * (z_diff ** 2)
    distancia_acustica = float(np.sqrt(np.sum(diff_quadrada_ponderada) / np.sum(w_vec)))
    score_aderencia = float(np.clip(100.0 * np.exp(-0.35 * distancia_acustica), 5.0, 99.0))

    # 4. Relatório Executivo Textual A&R
    print("
" + "█"*75)
    print(f"   🎧 DIAGNÓSTICO EXECUTIVO DE A&R — GÊNERO: {genero_alvo.upper()}")
    print("█"*75)
    print(f"• SCORE DE ADERÊNCIA AO MERCADO : {score_aderencia:.1f}% (Ponderado via SHAP)")
    if pesos_dict:
        print(f"• PESOS SHAP DO GÊNERO ({genero_alvo.upper()}): {pesos_dict}")
    print(f"• PROBABILIDADE DE HIT (P90)    : {hit_prob*100:.1f}%")
    print(f"• TEMPO / ANDAMENTO (BPM)       : {dsp['tempo']} BPM (Média Hits: {medias_hit['tempo']:.1f})")
    print(f"• DURAÇÃO TOTAL                 : {dsp['duration_min']} min ({struct['duration_s']:.0f}s)")
    print("-" * 75)
    
    print("📋 DIRETRIZES DE MASTERIZAÇÃO & STREAMING (EBU R128):")
    print(f"  [i] Loudness Integrado : {psy['integrated_lufs']} LUFS")
    if psy['integrated_lufs'] > -13.0:
        print(f"      ⚠️ ATENÇÃO: A faixa está {abs(psy['spotify_gain_change'])} dB mais alta que o padrão Spotify (-14 LUFS). Sofrerá atenuação algorítmica.")
    elif psy['integrated_lufs'] < -15.5:
        print(f"      ⚠️ ATENÇÃO: Faixa com volume baixo ({psy['integrated_lufs']} LUFS). O Spotify aplicará limiter/ganho positivo.")
    else:
        print("      ✅ EXCELENTE: Volume perfeitamente calibrado para as plataformas de streaming (-14 LUFS).")

    print(f"  [i] True Peak (dBTP)   : {psy['true_peak_dbtp']} dBTP " + ("(✅ Seguro)" if psy['true_peak_dbtp'] <= -1.0 else "(⚠️ Risco de clipping inter-amostral)"))
    print(f"  [i] Crest Factor       : {psy['crest_factor_db']} dB (Faixa dinâmica de micro-compressão)")

    print("
⏱️ ESTRUTURA TEMPORAL & RETENÇÃO DE HOOK:")
    print(f"  [i] Tempo até o 1º Refrão : {struct['time_to_hook_s']}s")
    if struct['time_to_hook_s'] > 50.0:
        print("      ⚠️ RECOMENDAÇÃO A&R: O 1º Refrão/Drop demora para chegar (> 50s). Considere encurtar a intro para reduzir o skip rate.")
    else:
        print("      ✅ Rápida retenção: O Hook surge nos primeiros 50s, ideal para engajamento em playlists.")
    print(f"  [i] Dynamic Lift (Surge)  : +{struct['dynamic_lift_pct']}% de impacto no Refrão")

    print("█"*75)

    # 5. Renderização do Dashboard Visual 4-em-1
    fig = plt.figure(figsize=(16, 12))

    # Painel 1: Radar Polar de Assinatura Acústica
    ax1 = fig.add_subplot(2, 2, 1, projection='polar')
    radar_cols = ['danceability', 'energy', 'valence', 'acousticness', 'speechiness']
    user_radar = [dsp[c] for c in radar_cols] + [dsp[radar_cols[0]]]
    hit_radar = [medias_hit[c] for c in radar_cols] + [medias_hit[radar_cols[0]]]
    angles = np.linspace(0, 2 * np.pi, len(radar_cols), endpoint=False).tolist()
    angles += angles[:1]

    ax1.plot(angles, user_radar, color=SPOTIFY_GREEN, linewidth=2.5, label='Sua Faixa')
    ax1.fill(angles, user_radar, color=SPOTIFY_GREEN, alpha=0.30)
    ax1.plot(angles, hit_radar, color=SPOTIFY_ACCENT, linewidth=2, linestyle='--', label=f'Média Hits ({genero_alvo})')
    ax1.fill(angles, hit_radar, color=SPOTIFY_ACCENT, alpha=0.15)
    ax1.set_ylim(0, 1.0)
    ax1.set_thetagrids(np.degrees(angles[:-1]), ['Dançabilidade', 'Energia', 'Valência', 'Acústica', 'Speechiness'], fontweight='bold')
    ax1.set_title("A. Assinatura Acústica (Radar)", fontsize=12, pad=15, fontweight='bold')
    ax1.legend(loc='upper right', bbox_to_anchor=(1.25, 1.15))

    # Painel 2: Balanço Tonal por Frequências (Match EQ)
    ax2 = fig.add_subplot(2, 2, 2)
    bands = list(psy['band_energies'].keys())
    energies = list(psy['band_energies'].values())
    x_pos = np.arange(len(bands))
    ax2.bar(x_pos, energies, color='#3b5998', width=0.5, edgecolor='white', alpha=0.85)
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(bands, rotation=15, ha='right', fontsize=9)
    ax2.set_ylabel('Distribuição de Energia (%)')
    ax2.set_title("B. Balanço Tonal em 5 Bandas (Match EQ)", fontsize=12, fontweight='bold')

    # Painel 3: Timeline de Energia & Detecção do 1º Hook
    ax3 = fig.add_subplot(2, 2, 3)
    ax3.plot(struct['timeline_times'], struct['timeline_rms'], color=SPOTIFY_GREEN, lw=1.5, label='Envelope de Dinâmica')
    ax3.axvline(struct['time_to_hook_s'], color=SPOTIFY_ACCENT, lw=2, linestyle='--', label=f"1º Hook / Drop ({struct['time_to_hook_s']}s)")
    ax3.set_xlabel('Tempo (segundos)')
    ax3.set_ylabel('Intensidade Relativa')
    ax3.set_title("C. Macro-Dinâmica Temporal & Retenção de Hook", fontsize=12, fontweight='bold')
    ax3.legend(loc='upper right')

    # Painel 4: Gap Analysis das Features Acústicas
    ax4 = fig.add_subplot(2, 2, 4)
    gap_cols = ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'valence']
    gaps = [(dsp[c] - medias_hit[c]) for c in gap_cols]
    colors = [SPOTIFY_GREEN if g >= 0 else '#e74c3c' for g in gaps]
    ax4.barh(gap_cols, gaps, color=colors, edgecolor='black', alpha=0.8)
    ax4.axvline(0, color='gray', linestyle='--')
    ax4.set_xlabel('Discrepância vs Média dos Hits (Gap)')
    ax4.set_title("D. Gap Analysis Acústico vs Sucessos do Gênero", fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.show()

print("✅ Motor A&R Completo compilado com sucesso!")


<a id='sec9'></a>
## 9. Simulador Interativo 'What-If' de Produção

O simulador permite que o produtor teste hipóteses de arranjo ou remix (ex: acelerar BPM, aumentar peso de graves, comprimir mixagem) e visualize a probabilidade recalculada em tempo real.


In [ ]:
def simular_what_if(danceability, energy, loudness, speechiness, acousticness, instrumentalness, liveness, valence, tempo, genero_alvo='pop'):
    """
    Simula ajustes em parâmetros de produção e recalcula a aderência técnica e probabilidade P90.
    """
    sim_data = pd.DataFrame([{
        'danceability': danceability,
        'energy': energy,
        'loudness': loudness,
        'speechiness': speechiness,
        'acousticness': acousticness,
        'instrumentalness': instrumentalness,
        'liveness': liveness,
        'valence': valence,
        'tempo': tempo
    }])[ACOUSTIC_FEATURES]

    prob = model.predict_proba(sim_data)[0, 1]
    
    benchmark = df_unique[(df_unique['track_genre'] == genero_alvo) & (df_unique['is_hit'] == 1)]
    if len(benchmark) == 0:
        benchmark = df_unique[df_unique['is_hit'] == 1]
    
    medias_hit = benchmark[ACOUSTIC_FEATURES].mean()
    desvios_hit = benchmark[ACOUSTIC_FEATURES].std().replace(0, 1.0)
    z_diff = (sim_data.iloc[0] - medias_hit) / desvios_hit
    aderencia = float(np.clip(100.0 * np.exp(-0.35 * np.sqrt(np.sum(z_diff ** 2))), 5.0, 99.0))

    print("="*60)
    print(f"🔮 SIMULAÇÃO WHAT-IF (Cenário: {genero_alvo.upper()})")
    print("="*60)
    print(f"• Probabilidade de Hit Estimada : {prob*100:.1f}%")
    print(f"• Aderência Técnica ao Mercado  : {aderencia:.1f}%")
    print("="*60)

# Demonstração de Teste do Simulador
simular_what_if(
    danceability=0.72, energy=0.85, loudness=-5.5,
    speechiness=0.06, acousticness=0.15, instrumentalness=0.01,
    liveness=0.12, valence=0.68, tempo=124.0,
    genero_alvo='pop'
)


<a id='sec10'></a>
## 10. Interface de Upload & Demonstração com Áudio Real/Sintético

Se estiver no Google Colab, você pode fazer upload de um arquivo `.mp3` ou `.wav` real.
Caso contrário, uma faixa de teste sintética de alta fidelidade é sintetizada automaticamente para validação do pipeline fim a fim.


In [ ]:
def gerar_faixa_teste_sintetica(filename='faixa_teste_synth.wav', duration=35, sr=22050):
    """
    Gera uma faixa de áudio sintética com estrutura musical (harmonia, bumbo, melodia)
    para permitir teste autônomo sem necessidade imediata de upload externo.
    """
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    
    # 1. Bumbo / Kick 4-on-the-floor (120 BPM = batida a cada 0.5s)
    beat_interval = 0.5
    kick = np.zeros_like(t)
    for b in np.arange(0, duration, beat_interval):
        idx = int(b * sr)
        kick_len = int(0.15 * sr)
        if idx + kick_len < len(t):
            t_k = np.linspace(0, 0.15, kick_len)
            kick[idx:idx+kick_len] += np.sin(2 * np.pi * 55 * np.exp(-t_k * 25) * t_k) * np.exp(-t_k * 15)

    # 2. Acordes de Sintetizador (Dó Maior -> Lá Menor -> Fá Maior -> Sol Maior)
    chord_freqs = [261.63, 329.63, 392.00] # Acorde C Maior
    synth = sum(0.15 * np.sin(2 * np.pi * f * t) for f in chord_freqs)

    # 3. Seção Drop / Refrão aos 14 segundos (Surge dinâmico)
    dynamic_envelope = np.where(t < 14.0, 0.4, 0.95)
    
    signal_final = (kick * 0.6 + synth * 0.4) * dynamic_envelope
    signal_final = signal_final / np.max(np.abs(signal_final) + 1e-6) * 0.85
    
    sf.write(filename, signal_final, sr)
    print(f"🎵 Faixa sintetizada de teste salva em: {filename}")
    return filename

# Execução do Teste Integrado
arquivo_demo = 'faixa_teste_synth.wav'
gerar_faixa_teste_sintetica(arquivo_demo)

# Diagnóstico da faixa demo comparada com Rock e Pop
diagnosticar_faixa_a_e_r(
    audio_path=arquivo_demo,
    genero_alvo='rock',
    model=model,
    df_base=df_unique,
    features=ACOUSTIC_FEATURES,
    genre_weights=genre_shap_weights
)
